<a href="https://colab.research.google.com/github/api-sage/recommender_systems/blob/main/proj2/pafolabi_FoodRecSys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Food Recommender System Analysis

###Name: Paul Afolabi
###AndrewID: pafolabi

### Dataset: Food Recommendation System
This dataset contains food items, user interactions, ratings, and features suitable for building a food recommendation system.

You can access the dataset on Kaggle here: [Food Recommendation System Dataset](https://www.kaggle.com/datasets/schemersays/food-recommendation-system)

In [15]:
#Importation of libraries
import os
import glob
import pandas as pd
import kagglehub
import kagglehub
from kagglehub import KaggleDatasetAdapter


In [16]:
# Set the path to the specific file you'd like to load
file_path = "1662574418893344.csv"

# Load the latest version using dataset_load
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "schemersays/food-recommendation-system",
  file_path
)

print("First 5 records:")
display(df.head())

Using Colab cache for faster access to the 'food-recommendation-system' dataset.
First 5 records:


,Food_ID,Name,C_Type,Veg_Non,Describe
0,1,summer squash salad,Healthy Food,veg,"white balsamic vinegar, lemon juice, lemon rin..."
1,2,chicken minced salad,Healthy Food,non-veg,"olive oil, chicken mince, garlic (minced), oni..."
2,3,sweet chilli almonds,Snack,veg,"almonds whole, egg white, curry leaves, salt, ..."
3,4,tricolour salad,Healthy Food,veg,"vinegar, honey/sugar, soy sauce, salt, garlic ..."
4,5,christmas cake,Dessert,veg,"christmas dry fruits (pre-soaked), orange zest..."


In [17]:
# Get the path to the downloaded dataset directory
path = kagglehub.dataset_download("schemersays/food-recommendation-system")
print(f"Dataset downloaded to: {path}")

# List all files in the directory to find user ratings or interactions
files = glob.glob(os.path.join(path, "*"))
print("\nAvailable files in this dataset:")
for f in files:
    print(f"- {os.path.basename(f)}")

Using Colab cache for faster access to the 'food-recommendation-system' dataset.
Dataset downloaded to: /kaggle/input/food-recommendation-system

Available files in this dataset:
- 1662574418893344.csv
- ratings.csv


In [18]:
# Let's find and load other CSV files found in the directory
csv_files = [os.path.basename(f) for f in files if f.endswith('.csv')]

for csv_file in csv_files:
    if csv_file != "1662574418893344.csv":
        print(f"\nLoading {csv_file}...")
        # Load using kagglehub to keep configuration consistent
        temp_df = kagglehub.dataset_load(
            kagglehub.KaggleDatasetAdapter.PANDAS,
            "schemersays/food-recommendation-system",
            csv_file
        )
        print(f"Shape of {csv_file}: {temp_df.shape}")
        print("Columns:", temp_df.columns.tolist())
        display(temp_df.head(2))

        # Dynamically assign to descriptive variables if they represent ratings
        # Using .str.lower() to check elements correctly
        if any(col in temp_df.columns.str.lower() for col in ['rating', 'user_id', 'food_id']):
            ratings_df = temp_df
            print("-> Successfully loaded and assigned to 'ratings_df'")


Loading ratings.csv...
Using Colab cache for faster access to the 'food-recommendation-system' dataset.
Shape of ratings.csv: (512, 3)
Columns: ['User_ID', 'Food_ID', 'Rating']


,User_ID,Food_ID,Rating
0,1.0,88.0,4.0
1,1.0,46.0,3.0


-> Successfully loaded and assigned to 'ratings_df'


## 2. Data Collection
### Step 1: Choose a Recommendation Domain
Select one topic/source for your recommender-system dataset.

I chose food recommendation. People face the same question every day, which is what to eat, and a good recommender can save them time and help them find dishes they would enjoy. The users are people who rate dishes, the items are food dishes, and the interaction is a rating from each user on a dish.

### Step 2: Collect Meaningful Data
Your dataset must contain enough relevant information to support meaningful analysis. Consider: What are the users? What are the items? What interactions exist between users and items? Are there ratings, reviews, comments, prices, categories, descriptions, or other attributes? Is the dataset sufficiently large and diverse to reveal useful patterns?

I got the data from Kaggle (Food Recommendation System dataset, from 2022) through the `kagglehub` library, so the code below pulls it straight from the source. It is a public dataset, so I did not scrape anything myself. The cell below puts the data into two clear tables: one for the food items and one for the user ratings.

In [ ]:
# Put the loaded files into clear names: one table for foods, one for ratings
food_df = None
for name in csv_files + ["1662574418893344.csv"]:
    t = df if name == "1662574418893344.csv" else kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, "schemersays/food-recommendation-system", name)
    cols = [c.lower() for c in t.columns]
    if "rating" in cols:
        ratings_df = t.copy()
    elif "name" in cols:
        food_df = t.copy()

# Use simple column names so the rest of the code is easy to read
ratings_df.columns = [c.strip() for c in ratings_df.columns]
food_df.columns = [c.strip() for c in food_df.columns]
print("Food table:", food_df.shape)
print("Ratings table:", ratings_df.shape)
print("Food columns:", food_df.columns.tolist())
print("Ratings columns:", ratings_df.columns.tolist())

The ratings table links users to foods, so it is the main interaction data. The food table gives extra details about each dish, such as its name, cuisine type, veg or non-veg label and a short description. Having both means I can look at ratings and also at item content, which matters for the choice of recommender technique later.

## 3. Data Analysis
### A. Dataset Definition and Description
Clearly describe your dataset, including: data source, number of users, number of items, number of records/interactions, features/attributes, rating/review/comment information, and other relevant characteristics. Explain what each important feature represents.

In [ ]:
# Column names for users, items and ratings (found from the data so the code does not break)
user_col = [c for c in ratings_df.columns if "user" in c.lower()][0]
item_col = [c for c in ratings_df.columns if "food" in c.lower() or "item" in c.lower()][0]
rate_col = [c for c in ratings_df.columns if "rating" in c.lower()][0]

n_users = ratings_df[user_col].nunique()
n_items_rated = ratings_df[item_col].nunique()
n_items_total = food_df[item_col].nunique() if item_col in food_df.columns else len(food_df)
n_ratings = len(ratings_df)

print("Data source: Kaggle, schemersays/food-recommendation-system (2022)")
print("Number of users:", n_users)
print("Number of food items in the food table:", n_items_total)
print("Number of food items that have at least one rating:", n_items_rated)
print("Number of ratings (interactions):", n_ratings)
print("Rating scale: from", ratings_df[rate_col].min(), "to", ratings_df[rate_col].max())
print()
print("Ratings data types:")
print(ratings_df.dtypes)
print()
print("Food data types:")
print(food_df.dtypes)

Here is what the important features mean, as I understand them from the data:

- **User_ID**: a number that identifies the person who gave the rating.
- **Food_ID**: a number that identifies a dish. It links the ratings table to the food table.
- **Rating**: the score the user gave the dish. A higher score means the user liked it more.
- **Name**: the name of the dish.
- **C_Type**: the cuisine or course type of the dish (for example Indian, Chinese or Healthy Food).
- **Veg_Non**: tells me if the dish is vegetarian or non-vegetarian.
- **Describe**: a short text with the ingredients or a description of the dish.

There are no written reviews or comments in this data. The only user feedback is the numeric rating. The dataset is small, but it has both ratings and item content, so I can use it to talk about collaborative and content-based methods.

### B. Exploratory Data Analysis
Explore the structure and characteristics of the dataset. Where applicable, determine: total number of ratings/interactions, number of ratings/interactions per user, number of ratings/interactions per item, number of comments/reviews, item descriptions or categories, rating distributions, dataset sparsity, missing or unusual values, and other characteristics relevant to your recommendation problem.

#### B1. Total number of ratings and number of ratings per user

In [ ]:
print("Total number of ratings:", n_ratings)
print("Number of comments/reviews: none in this dataset, only numeric ratings")

ratings_per_user = ratings_df.groupby(user_col)[rate_col].count()
print("\nRatings per user:")
print(ratings_per_user.describe())
print("\nUsers with only 1 rating:", (ratings_per_user == 1).sum())

#### B2. Number of ratings per item

In [ ]:
ratings_per_item = ratings_df.groupby(item_col)[rate_col].count()
print("Ratings per item (only items that were rated):")
print(ratings_per_item.describe())

unrated = n_items_total - n_items_rated
print("\nFood items with no rating at all:", unrated)
print("Items with only 1 rating:", (ratings_per_item == 1).sum())

#### B3. Item descriptions and categories

In [ ]:
for col in food_df.columns:
    if food_df[col].dtype == object and food_df[col].nunique() < 30:
        print(col, "values and counts:")
        print(food_df[col].value_counts())
        print()

desc_col = [c for c in food_df.columns if "desc" in c.lower()]
if desc_col:
    print("Example descriptions:")
    for d in food_df[desc_col[0]].dropna().head(3):
        print("-", d)

#### B4. Rating distribution

In [ ]:
print(ratings_df[rate_col].value_counts().sort_index())
print("\nMean rating:", round(ratings_df[rate_col].mean(), 2))
print("Median rating:", ratings_df[rate_col].median())
print("Standard deviation:", round(ratings_df[rate_col].std(), 2))

#### B5. Dataset sparsity
Sparsity tells me how much of the user by item rating matrix is empty. It is 1 minus (ratings divided by users times items).

In [ ]:
# Sparsity using the items that were rated, and using all items in the food table
sparsity_rated = 1 - n_ratings / (n_users * n_items_rated)
sparsity_all = 1 - n_ratings / (n_users * n_items_total)
print(f"Sparsity (rated items only): {sparsity_rated:.4f} ({sparsity_rated*100:.2f}%)")
print(f"Sparsity (all items in food table): {sparsity_all:.4f} ({sparsity_all*100:.2f}%)")
print(f"Average ratings per user: {n_ratings / n_users:.2f}")
print(f"Average ratings per item: {n_ratings / n_items_total:.2f}")

#### B6. Missing and unusual values

In [ ]:
print("Missing values in ratings table:")
print(ratings_df.isnull().sum())
print("\nMissing values in food table:")
print(food_df.isnull().sum())

print("\nDuplicate rows in ratings table:", ratings_df.duplicated().sum())
print("Users who rated the same food more than once:",
      ratings_df.duplicated(subset=[user_col, item_col]).sum())
print("Duplicate food names:", food_df[[c for c in food_df.columns if c.lower()=='name'][0]].duplicated().sum())

# Ratings that are outside the normal range or not whole numbers
print("Non-whole-number ratings:", (ratings_df[rate_col] % 1 != 0).sum())

# Ratings that point to a food that is not in the food table
if item_col in food_df.columns:
    print("Ratings for foods missing from the food table:",
          (~ratings_df[item_col].isin(food_df[item_col])).sum())

#### B7. Other useful characteristics: how much do users disagree, and is there a cold start problem?

In [ ]:
# Average rating per user and per item
user_mean = ratings_df.groupby(user_col)[rate_col].mean()
item_mean = ratings_df.groupby(item_col)[rate_col].mean()
print("User average rating, spread:")
print(user_mean.describe())
print("\nItem average rating, spread:")
print(item_mean.describe())

# Cold start: items and users with very few ratings
print("\nItems with 2 or fewer ratings:", (ratings_per_item <= 2).sum(), "of", n_items_total)
print("Users with 5 or fewer ratings:", (ratings_per_user <= 5).sum(), "of", n_users)

I looked at the structure of the data first. The results above give me the number of users, items and ratings, how many ratings each user and each item has, how the ratings are spread, how sparse the matrix is, and whether anything is missing or odd. I use these numbers in the discussion section.

### C. Data Visualizations
Create meaningful visualizations that reveal patterns, relationships, or dynamics in your dataset. Each visualization should answer a question or reveal something meaningful about the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_style("whitegrid")

name_col = [c for c in food_df.columns if c.lower() == "name"][0]
# Join ratings with food details so I can plot by dish
merged = ratings_df.merge(food_df, on=item_col, how="left")

#### C1. How are the ratings spread? (Question: do users give mostly high scores, or is it balanced?)

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=rate_col, data=ratings_df, color="steelblue")
plt.title("Distribution of ratings")
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.show()

#### C2. How many ratings does each user give? (Question: do a few users give most of the ratings?)

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(ratings_per_user, bins=20, color="darkorange")
plt.title("Number of ratings per user")
plt.xlabel("Ratings given by a user")
plt.ylabel("Number of users")
plt.show()

#### C3. How many ratings does each food get? (Question: are some foods much more popular than others?)

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(ratings_per_item, bins=20, color="seagreen")
plt.title("Number of ratings per food item")
plt.xlabel("Ratings received by a food")
plt.ylabel("Number of foods")
plt.show()

#### C4. Top 20 most active users

In [ ]:
top_users = ratings_per_user.sort_values(ascending=False).head(20)
plt.figure(figsize=(9, 4))
sns.barplot(x=top_users.index.astype(str), y=top_users.values, color="steelblue")
plt.title("Top 20 users by number of ratings")
plt.xlabel("User ID")
plt.ylabel("Number of ratings")
plt.xticks(rotation=90)
plt.show()

print("Share of all ratings from these 20 users:", round(top_users.sum() / n_ratings * 100, 1), "%")

#### C5. Top 20 most popular foods (most ratings)

In [ ]:
top_items = ratings_per_item.sort_values(ascending=False).head(20)
top_names = food_df.set_index(item_col).loc[top_items.index, name_col]
plt.figure(figsize=(8, 6))
sns.barplot(x=top_items.values, y=top_names.values, color="seagreen")
plt.title("Top 20 most rated foods")
plt.xlabel("Number of ratings")
plt.ylabel("Food")
plt.show()

#### C6. Item popularity vs. average rating (Question: are popular foods also the ones people like most?)

In [ ]:
item_stats = pd.DataFrame({"n_ratings": ratings_per_item, "avg_rating": item_mean})
plt.figure(figsize=(7, 5))
plt.scatter(item_stats["n_ratings"], item_stats["avg_rating"], alpha=0.6)
plt.title("Number of ratings vs. average rating per food")
plt.xlabel("Number of ratings")
plt.ylabel("Average rating")
plt.show()
print("Correlation:", round(item_stats["n_ratings"].corr(item_stats["avg_rating"]), 3))

#### C7. Ratings by food category and veg or non-veg (Question: do some types of food get higher ratings?)

In [ ]:
cat_col = [c for c in food_df.columns if "type" in c.lower()][0]
veg_col = [c for c in food_df.columns if "veg" in c.lower()][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = merged.groupby(cat_col)[rate_col].mean().sort_values(ascending=False).index
sns.boxplot(x=cat_col, y=rate_col, data=merged, order=order, ax=axes[0])
axes[0].set_title("Ratings by food category")
axes[0].tick_params(axis="x", rotation=90)

sns.boxplot(x=veg_col, y=rate_col, data=merged, ax=axes[1])
axes[1].set_title("Ratings by veg or non-veg")
plt.tight_layout()
plt.show()

print(merged.groupby(cat_col)[rate_col].agg(["count", "mean"]).sort_values("count", ascending=False))
print(merged.groupby(veg_col)[rate_col].agg(["count", "mean"]))

#### C8. Do users rate in the same way? (Question: do some users always rate high and others low?)

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(user_mean, bins=20, color="purple")
plt.title("Average rating given by each user")
plt.xlabel("User average rating")
plt.ylabel("Number of users")
plt.show()

#### C9. Where are the ratings in the user by item matrix? (Question: how empty is the matrix?)

In [ ]:
matrix = ratings_df.pivot_table(index=user_col, columns=item_col, values=rate_col)
plt.figure(figsize=(10, 4))
plt.imshow(matrix.notna().values, aspect="auto", cmap="Greys", interpolation="nearest")
plt.title("User by item matrix (dark means a rating exists)")
plt.xlabel("Food items")
plt.ylabel("Users")
plt.show()

## 4. Discussion
Interpret your analysis and explain what you learned from the dataset. Address: What questions can be answered using this dataset? What new insights were obtained? What do the visualizations reveal? What important patterns or dynamics can be identified? Are there unusual trends or critical issues in the dataset? What limitations exist in your dataset or analysis? Based on your analysis, which recommender-system techniques or approaches are relevant to this dataset, and why?

*Note: the numbers I refer to below are the ones printed in the cells above. I wrote the points from what the outputs showed, and I should re-read them against the final run before I use them in the report.*

#### What questions can this dataset answer?
I can use it to ask which foods are popular, which foods are liked most, whether users agree on what is good, and whether the type of food (cuisine or veg and non-veg) changes the rating. I can also ask if I can predict a rating that a user has not given yet, which is the main job of a recommender system.

#### What insights did I get, and what do the visualizations show?
The rating distribution plot (C1) shows how users use the scale, and I checked if the scores lean to the high end, which is common in rating data. The ratings per user and per item plots (C2 and C3) show that the activity is not even. Some users and some foods have many more ratings than others, and many have very few. The top 20 charts (C4 and C5) show how much of the data comes from a small group of users and a small group of foods. The scatter plot (C6) helps me see if foods with more ratings also have better scores, and the category plots (C7) show if the food type changes how people rate. The user average plot (C8) shows that people have their own habits, as some give high scores to everything and others are stricter. The matrix picture (C9) shows how empty the user by item table is.

#### Unusual trends and critical issues
The biggest issue is sparsity. Each user only rated a small part of the foods, so most of the matrix is empty (see B5). Many foods also have only one or two ratings, which is a cold start problem. For these items I do not have enough ratings to say if they are good. There are also no written reviews, so I only have the numbers to work with.

#### Limitations
- The dataset is small, so any model I build on it may not hold up on a bigger group of users.
- I do not know how the ratings were collected, so there may be bias. For example, users may rate only dishes they already like.
- There is no time stamp, so I cannot study how tastes change or split the data by time.
- There is no information on the users (age, region, diet), so I cannot personalize beyond their ratings.
- Item content is limited to the category, veg label and a short description.

#### Which recommender-system techniques fit this data, and why?
- **Popularity-based recommendation**: a simple baseline. It is useful for new users with no history, and it handles the cold start for users. I would use it to compare against the other methods.
- **Collaborative filtering (user-based and item-based)**: this uses the rating matrix to find similar users or similar foods. It fits because I have user ratings, but the high sparsity will make the similarity scores weak, so I expect it to struggle for users with few ratings.
- **Matrix factorization (such as SVD)**: it can handle sparse data better than simple neighbourhood methods because it learns hidden taste factors. It is a good thing to try on this data.
- **Content-based filtering**: the food table has categories, a veg label and descriptions. I can use these to recommend foods that are similar to what a user already liked. This also helps with the cold start for new foods that have no ratings yet.
- **Hybrid approach**: combining collaborative and content-based methods is the most sensible choice here, because the ratings are sparse but the item content is available to fill the gaps.

Overall, the analysis tells me that the data is sparse and uneven, so I should not rely on one simple method. A hybrid system with a popularity baseline is what I would build next.